In [9]:
import pandas as pd
import numpy as np
import duckdb
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

In [10]:
DATA_PATH = Path("../data/online_retail_II.xlsx")
OUTPUT_PATH = Path("../outputs/retail_clean.parquet")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), f"Data file not found: {DATA_PATH.resolve()}"

---
## Section 1 — Load

Load both Excel sheets and concatenate into a single DataFrame.  
`Customer ID` is read as `float64` because it contains nulls (cannot be int until after cleaning).

In [11]:
df09 = pd.read_excel(DATA_PATH, sheet_name="Year 2009-2010", dtype={"Customer ID": "float64"})
df10 = pd.read_excel(DATA_PATH, sheet_name="Year 2010-2011", dtype={"Customer ID": "float64"})
df = pd.concat([df09, df10], ignore_index=True)

print(f"Sheet 1 (2009-2010) : {len(df09):>10,} rows")
print(f"Sheet 2 (2010-2011) : {len(df10):>10,} rows")
print(f"Combined            : {len(df):>10,} rows")

Sheet 1 (2009-2010) :    525,461 rows
Sheet 2 (2010-2011) :    541,910 rows
Combined            :  1,067,371 rows


In [12]:
print(f"Total rows  : {len(df):,}")
print(f"Date range  : {df['InvoiceDate'].min()}  →  {df['InvoiceDate'].max()}")
print(f"\nColumns and dtypes:")
print(df.dtypes)

Total rows  : 1,067,371
Date range  : 2009-12-01 07:45:00  →  2011-12-09 12:50:00

Columns and dtypes:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object


In [13]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


---
## Section 2 — Validate

Use DuckDB to count data quality issues:
- Nulls per column
- Rows where `Quantity <= 0` (returns / cancellations)
- Rows where `Price <= 0` (free samples, adjustments)
- Rows where `Customer ID` is null (guest transactions)

In [14]:
con = duckdb.connect()
con.register("raw", df)

quality = con.execute("""
    SELECT
        COUNT(*)                                                            AS total_rows,
        SUM(CASE WHEN "Customer ID" IS NULL THEN 1 ELSE 0 END)            AS null_customer_id,
        SUM(CASE WHEN Quantity <= 0       THEN 1 ELSE 0 END)              AS qty_lte_zero,
        SUM(CASE WHEN Price    <= 0       THEN 1 ELSE 0 END)              AS price_lte_zero,
        COUNT(*) - COUNT(
            DISTINCT Invoice
                  || StockCode
                  || CAST(InvoiceDate    AS VARCHAR)
                  || CAST("Customer ID" AS VARCHAR)
        )                                                                  AS approx_duplicates
    FROM raw
""").df()

In [16]:
# Null counts per column via DuckDB UNPIVOT
null_counts = con.execute("""
    WITH counts AS (
        SELECT
            SUM(CASE WHEN Invoice       IS NULL THEN 1 ELSE 0 END) AS Invoice,
            SUM(CASE WHEN StockCode     IS NULL THEN 1 ELSE 0 END) AS StockCode,
            SUM(CASE WHEN Description   IS NULL THEN 1 ELSE 0 END) AS Description,
            SUM(CASE WHEN Quantity      IS NULL THEN 1 ELSE 0 END) AS Quantity,
            SUM(CASE WHEN InvoiceDate   IS NULL THEN 1 ELSE 0 END) AS InvoiceDate,
            SUM(CASE WHEN Price         IS NULL THEN 1 ELSE 0 END) AS Price,
            SUM(CASE WHEN "Customer ID" IS NULL THEN 1 ELSE 0 END) AS CustomerID,
            SUM(CASE WHEN Country       IS NULL THEN 1 ELSE 0 END) AS Country
        FROM raw
    )
    SELECT column_name, null_count
    FROM counts
    UNPIVOT(null_count FOR column_name IN (
        Invoice, StockCode, Description, Quantity, InvoiceDate, Price, CustomerID, Country
    ))
    ORDER BY null_count DESC, column_name
""").df()

In [17]:
total       = quality["total_rows"][0]
null_cid    = quality["null_customer_id"][0]
qty_bad     = quality["qty_lte_zero"][0]
price_bad   = quality["price_lte_zero"][0]
approx_dups = quality["approx_duplicates"][0]

print("=" * 45)
print("        DATA QUALITY SUMMARY")
print("=" * 45)
print(f"  Total rows           : {total:>10,}")
print(f"  Null Customer ID     : {null_cid:>10,}  ({null_cid/total*100:.1f}%)")
print(f"  Quantity <= 0        : {qty_bad:>10,}  ({qty_bad/total*100:.1f}%)")
print(f"  Price <= 0           : {price_bad:>10,}  ({price_bad/total*100:.1f}%)")
print(f"  Approx duplicates    : {approx_dups:>10,}  ({approx_dups/total*100:.1f}%)")
print("=" * 45)
print()
print("Null counts per column:")
print(null_counts.to_string(index=False))

        DATA QUALITY SUMMARY
  Total rows           :  1,067,371
  Null Customer ID     :  243,007.0  (22.8%)
  Quantity <= 0        :   22,950.0  (2.2%)
  Price <= 0           :    6,207.0  (0.6%)
  Approx duplicates    :    280,170  (26.2%)

Null counts per column:
column_name  null_count
 CustomerID    243007.0
Description      4382.0
    Country         0.0
    Invoice         0.0
InvoiceDate         0.0
      Price         0.0
   Quantity         0.0
  StockCode         0.0


In [18]:
fig = px.bar(
    null_counts[null_counts["null_count"] > 0],
    x="column_name",
    y="null_count",
    title="Null counts per column (raw data)",
    labels={"column_name": "Column", "null_count": "Null count"},
    text_auto=True,
)
fig.update_layout(showlegend=False)
fig.show()

---
## Section 3 — Clean

Filtering rules:
| Rule | Rationale |
|---|---|
| `Quantity > 0` | Remove returns and cancellations |
| `Price > 0` | Remove free samples and manual adjustments |
| `Customer ID` not null | Retain only identified customers (needed for CLV/churn) |
| Drop exact duplicates | Remove repeated rows from the raw export |

Type conversions applied after filtering:
- `InvoiceDate` → `datetime64`
- `Customer ID` → `int64`

In [19]:
rows_before = len(df)

df_clean = df.copy()
df_clean = df_clean[df_clean["Quantity"] > 0]
df_clean = df_clean[df_clean["Price"] > 0]
df_clean = df_clean[df_clean["Customer ID"].notna()]
df_clean = df_clean.drop_duplicates()

df_clean["InvoiceDate"] = pd.to_datetime(df_clean["InvoiceDate"])
df_clean["Customer ID"] = df_clean["Customer ID"].astype(int)

rows_after = len(df_clean)
removed    = rows_before - rows_after

print(f"Rows before cleaning : {rows_before:>10,}")
print(f"Rows after cleaning  : {rows_after:>10,}")
print(f"Rows removed         : {removed:>10,}  ({removed / rows_before * 100:.1f}%)")
print()
print("Final dtypes:")
print(df_clean.dtypes)

Rows before cleaning :  1,067,371
Rows after cleaning  :    779,425
Rows removed         :    287,946  (27.0%)

Final dtypes:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             int64
Country                   str
dtype: object


In [20]:
stages = ["Raw", "After Qty filter", "After Price filter", "After drop nulls", "After dedup"]
counts = [
    rows_before,
    int(df[df["Quantity"] > 0].pipe(len)),
    int(df[df["Quantity"] > 0][df[df["Quantity"] > 0]["Price"] > 0].pipe(len)),
    int(df[df["Quantity"] > 0][df[df["Quantity"] > 0]["Price"] > 0][
        df[df["Quantity"] > 0][df[df["Quantity"] > 0]["Price"] > 0]["Customer ID"].notna()
    ].pipe(len)),
    rows_after,
]

fig = px.funnel(
    x=counts,
    y=stages,
    title="Row counts through cleaning pipeline",
    labels={"x": "Rows", "y": "Stage"},
)
fig.show()

In [22]:
df_clean["StockCode"] = df_clean["StockCode"].astype("string")
df_clean.to_parquet(OUTPUT_PATH, engine="pyarrow", index=False)
print(f"Saved  → {OUTPUT_PATH.resolve()}")
print(f"Size   : {OUTPUT_PATH.stat().st_size / 1024 / 1024:.2f} MB")
print(f"Saved  → {OUTPUT_PATH.resolve()}")
print(f"Size   : {OUTPUT_PATH.stat().st_size / 1024 / 1024:.2f} MB")

Saved  → C:\Users\h11la\OneDrive\Documents\00 Portfolio\customer-analytics-ml-pipeline\retail-clv-churn-prediction\outputs\retail_clean.parquet
Size   : 4.88 MB
Saved  → C:\Users\h11la\OneDrive\Documents\00 Portfolio\customer-analytics-ml-pipeline\retail-clv-churn-prediction\outputs\retail_clean.parquet
Size   : 4.88 MB


In [23]:
# Smoke-check: reload and verify
check = pd.read_parquet(OUTPUT_PATH)

assert len(check) == rows_after,          "Row count mismatch after reload"
assert check["Customer ID"].dtype == int, "Customer ID should be int"
assert check["Quantity"].min() > 0,       "Quantity should all be > 0"
assert check["Price"].min() > 0,          "Price should all be > 0"
assert check["Customer ID"].isna().sum() == 0, "No nulls in Customer ID"

print(f"Reloaded rows : {len(check):,}")
print(f"Date range    : {check['InvoiceDate'].min()}  →  {check['InvoiceDate'].max()}")
print(f"\nDtypes:")
print(check.dtypes)
print("\nAll assertions passed.")

Reloaded rows : 779,425
Date range    : 2009-12-01 07:45:00  →  2011-12-09 12:50:00

Dtypes:
Invoice                 int64
StockCode              string
Description               str
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID             int64
Country                   str
dtype: object

All assertions passed.
